In [1]:
import pandas as pd
import os
import json
import datetime
import re

In [2]:
file_name = 'Final_Raw_data_HackerNews.json'
output_dir = '../raw_data'
os.makedirs(output_dir, exist_ok=True) 
output_path = os.path.join(output_dir, file_name)
with open(output_path, 'r', encoding='utf-8') as f:
        df = pd.DataFrame(json.load(f))

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36453 entries, 0 to 36452
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Title          36453 non-null  str  
 1   language       36453 non-null  str  
 2   Years          36453 non-null  str  
 3   Scraping_Date  36453 non-null  str  
dtypes: str(4)
memory usage: 1.1 MB


In [4]:
df['Scrap_Year'] = pd.to_datetime(df['Scraping_Date']).dt.year
time_info = df['Years'].str.extract(r'(\d+)\s+(\w+)\s+ago')
df['cleaned_year_comment'] = time_info[0].astype(float)
df['unit_year'] = time_info[1]
display(df.head())

,Title,language,Years,Scraping_Date,Scrap_Year,cleaned_year_comment,unit_year
0,Uv is the best thing to happen to the Python e...,Python,2214 points|todsacerdoti|4 months ago|1324 com...,2026-03-05 07:24:44.130929,2026,4.0,months
1,A from-scratch tour of Bitcoin in Python,Python,1208 points|yigitdemirag|5 years ago|277 comments,2026-03-05 07:24:44.296567,2026,5.0,years
2,Python 3.13 Gets a JIT,Python,1081 points|todsacerdoti|2 years ago|521 comments,2026-03-05 07:24:44.446665,2026,2.0,years
3,Show HN: I built a hardware processor that run...,Python,983 points|hwpythonner|10 months ago|265 comme...,2026-03-05 07:24:44.583786,2026,10.0,months
4,Prettymaps: Small Python library to draw custo...,Python,982 points|sebg|5 years ago|74 comments,2026-03-05 07:24:44.811589,2026,5.0,years


In [5]:
def get_actual_year(row):
    val = row['cleaned_year_comment']
    unit = str(row['unit_year'])
    scrap_year = row['Scrap_Year']
    scrap_month = pd.to_datetime(row['Scraping_Date']).month
    
    if pd.isna(val): 
        return scrap_year
    if 'year' in unit:
        return scrap_year - int(val)
    elif 'month' in unit:
        if (scrap_month - int(val)) <= 0: 
            return scrap_year - 1
        else:
            return scrap_year 
    else:
        return scrap_year

df['Actual_Post_Year'] = df.apply(get_actual_year, axis=1).astype(int)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36453 entries, 0 to 36452
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Title                 36453 non-null  str    
 1   language              36453 non-null  str    
 2   Years                 36453 non-null  str    
 3   Scraping_Date         36453 non-null  str    
 4   Scrap_Year            36453 non-null  int32  
 5   cleaned_year_comment  36453 non-null  float64
 6   unit_year             36453 non-null  str    
 7   Actual_Post_Year      36453 non-null  int64  
dtypes: float64(1), int32(1), int64(1), str(5)
memory usage: 2.1 MB


In [ ]:
import pandas as pd
import json
import re
import os

print("🚀 เริ่มกระบวนการ Clean Title แบบเข้มงวด...")

initial_rows = len(df)

rename_merge_map = {
    'Golang': 'Go',
    'R-lang': 'R',
    'Dlang': 'D',
    'Assembly language': 'Assembly',
    'PL/SQL': 'SQL',
    'Transact-SQL': 'SQL',
    'Classic Visual Basic': 'Visual Basic',
    'Delphi/Object Pascal': 'Delphi'
}
df['language'] = df['language'].replace(rename_merge_map)

def clean_title_text(text):
    text = re.sub(r'^(Show HN:|Ask HN:|Tell HN:|Launch HN:)\s*', '', str(text), flags=re.IGNORECASE)
    return text.strip()

df['clean_title'] = df['Title'].apply(clean_title_text)

def refine_c_family(title, current_lang):
    if current_lang not in ['C++', 'C']:
        return [current_lang]
        
    title_str = str(title)
    found = set()
    title_lower = title_str.lower()
    
    if re.search(r'C\+\+', title_str) or "c with classes" in title_lower:
        found.add('C++')
    
    cpp_fws = ['qt', 'unreal engine', 'boost', 'tauri']
    if any(re.search(r'\b' + fw + r'\b', title_lower) for fw in cpp_fws):
        found.add('C++')
        
    temp_title = re.sub(r'(?i)c with classes', '', title_str)
    c_pattern = r'(?<![a-zA-Z\-])\bC\b(?![#\+\-\w])'
    c_standard = r'\bC(89|99|11|17|23)\b'
    
    if re.search(c_pattern, temp_title) or re.search(c_standard, temp_title):
        found.add('C')
            
    
    return list(found) if found else []

df['language'] = df.apply(lambda row: refine_c_family(row['clean_title'], row['language']), axis=1)


df = df.explode('language')
df = df.dropna(subset=['language']).reset_index(drop=True)


keywords_map = {
    "Python": ["python", "django", "flask", "fastapi", "pytorch"],
    "Java": ["java", "spring boot", "hibernate", "struts"],
    "JavaScript": ["javascript", "js", "react", "vue", "node", "node.js", "next.js"],
    "C#": ["c#", ".net", "unity", "xamarin", "c sharp"],
    "PHP": ["php", "laravel", "symfony", "codeigniter"],
    "Visual Basic": ["visual basic", "vba", "vb.net", "vb6", ".net framework"],
    "SQL": ["sql", "postgresql", "mysql", "t-sql", "sql server", "oracle"],
    "Assembly": ["assembly", "nasm", "masm", "asm"],
    "Swift": ["swift", "swiftui", "uikit", "vapor"],
    "Rust": ["rust", "rocket", "actix", "warp"],
    "Fortran": ["fortran", "lapack", "scipy"],
    "Delphi": ["delphi", "vcl", "firemonkey", "pascal"],
    "MATLAB": ["matlab", "simulink"],
    "Kotlin": ["kotlin", "ktor", "spring boot", "jetpack", "android"],
    "Ruby": ["ruby", "rails", "sinatra"],
    "TypeScript": ["typescript", "ts", "angular", "nestjs", "svelte"],
    "COBOL": ["cobol", "mainframe", "cics"],
    "Scratch": ["scratch"],
    "Dart": ["dart", "flutter"],
    "Julia": ["julia", "flux", "genie"],
    "Lisp": ["lisp", "emacs"],
    "Scala": ["scala", "akka", "play", "spark"],
    "Prolog": ["prolog"],
    "Lua": ["lua", "löve", "roblox"],
    "Perl": ["perl", "catalyst", "dancer"],
    "Objective-C": ["objective-c", "cocoa", "foundation"],
    "Haskell": ["haskell", "yesod", "servant"],
    "VBScript": ["vbscript", "asp"],
    "F#": ["f#", "giraffe", "fable"],
    "Groovy": ["groovy", "grails", "gradle"],
    "Erlang": ["erlang", "otp"],
    "Elixir": ["elixir", "phoenix", "nerves"],
    "Clojure": ["clojure", "luminus", "ring"],
    "Smalltalk": ["smalltalk", "pharo", "seaside"],
    "Go": ["go", "golang", "gin", "echo", "fiber"],
    "R": ["r", "tidyverse", "shiny", "ggplot2", "r-lang"],
    "Ada": ["ada", "gnat"],
    "Logo": ["logo", "turtle"],
    "ABAP": ["abap", "sap"],
    "Scheme": ["scheme", "racket", "guile"],
    "D": ["d", "phobos", "vibe.d", "dlang"],
    "Apex": ["apex", "salesforce"],
    "Tcl": ["tcl", "tk"]
}

def is_relevant(row):
    title = str(row['clean_title']).lower()
    lang = str(row['language'])
    
    if lang in ['C', 'C++']:
        return True

    if lang == 'R':
        if any(kw in title for kw in ['r-lang', 'tidyverse', 'ggplot2', 'shiny']):
            return True
        if re.search(r'(?<![a-zA-Z0-9])r(?![a-zA-Z0-9&])', title) and "r&d" not in title:
            return True
        return False
    if lang == 'D':
        if any(kw in title for kw in ['dlang', 'phobos', 'vibe.d']):
            return True
        if re.search(r'(?<![a-zA-Z0-9])d(?![a-zA-Z0-9&])', title) and "3d" not in title and "2d" not in title and "d&d" not in title:
            return True
        return False

    if lang == 'Assembly':
        if any(kw in title for kw in ['nasm', 'masm', 'asm']):
            return True
        if "assembly" in title:
            if any(bad in title for bad in ["general assembly", "assembly line", "school assembly"]):
                return False
            return True
        return False
    keywords = keywords_map.get(lang, [lang.lower()])
    for kw in keywords:
        kw_lower = kw.lower()
        pattern = rf'(?<![a-zA-Z0-9]){re.escape(kw_lower)}(?![a-zA-Z0-9])'
        if re.search(pattern, title):
            return True
            
    return False

df = df[df.apply(is_relevant, axis=1)]

df_clean = df.drop_duplicates(subset=['clean_title', 'language'], keep='first')

dropped_total = initial_rows - len(df_clean)
print("DATA VALIDATION")
print("="*45)
print(f"Start Data: {initial_rows} row")
print(f"Deleted: {dropped_total} row")
print(f"End Data: {len(df_clean)} row")
print("="*45)

output_dir = '../raw_data'
output_file = 'Final_HackerNews_Cleaned.json'
output_path = os.path.join(output_dir, output_file)
final_df = df_clean[['clean_title','language','Scrap_Year','Actual_Post_Year']]
os.makedirs(output_dir, exist_ok=True) 
final_df.to_json(output_path, orient='records', force_ascii=False, indent=4)

print("\nFile saved!")

🚀 เริ่มกระบวนการ Clean Title แบบเข้มงวด...

 📊 DATA VALIDATION REPORT (ผลการตรวจสอบ)
🔹 ข้อมูลเริ่มต้น (ดิบ): 28055 แถว
🚨 ลบข้อมูลที่ซ้ำซ้อนและไม่มี Keyword ในชื่อกระทู้: 3558 แถว
✅ ข้อมูลสุทธิพร้อมใช้งานจริง: 24497 แถว

🎉 บันทึกไฟล์เสร็จสมบูรณ์ที่: ../raw_data\Final_HackerNews_Cleaned.json


In [10]:
final_df = df[['clean_title','language','Scrap_Year','Actual_Post_Year']]
file_name = 'Final_HackerNews_Cleaned.json'
output_dir = '../raw_data'
output_path = os.path.join(output_dir, file_name)
os.makedirs(output_dir, exist_ok=True)
final_df.to_json(output_path, orient='records', force_ascii=False, indent=4)

In [12]:
import pandas as pd
import json
import re
import os



initial_rows = len(df)

# 📌 2. ยุบรวมและเปลี่ยนชื่อภาษาให้เป็นมาตรฐาน (Normalization)
rename_merge_map = {
    'Golang': 'Go',
    'R-lang': 'R',
    'Dlang': 'D',
    'Assembly language': 'Assembly',
    'PL/SQL': 'SQL',
    'Transact-SQL': 'SQL',
    'Classic Visual Basic': 'Visual Basic',
    'Delphi/Object Pascal': 'Delphi'
}
df['language'] = df['language'].replace(rename_merge_map)

# 📌 3. แยก C/C++ และระเบิดแถว
def refine_c_family(title, current_lang):
    if current_lang not in ['C++', 'C']:
        return [current_lang]
        
    title_str = str(title)
    found = set()
    title_lower = title_str.lower()
    
    if re.search(r'C\+\+', title_str) or "c with classes" in title_lower:
        found.add('C++')
        
    cpp_fws = ['qt', 'unreal engine', 'boost', 'tauri']
    if any(re.search(r'\b' + fw + r'\b', title_lower) for fw in cpp_fws):
        found.add('C++')
        
    temp_title = re.sub(r'(?i)c with classes', '', title_str)
    c_pattern = r'(?<![a-zA-Z\-])\bC\b(?![#\+\-\w])'
    c_standard = r'\bC(89|99|11|17|23)\b'
    
    if re.search(c_pattern, temp_title) or re.search(c_standard, temp_title):
        found.add('C')
            
    return list(found) if found else ['C++']

df['language'] = df.apply(lambda row: refine_c_family(row['Title'], row['language']), axis=1)
df = df.explode('language').reset_index(drop=True)

# 📌 4. ทำความสะอาด Title (ลบขยะจาก Hacker News)
def clean_title_text(text):
    text = re.sub(r'^(Show HN:|Ask HN:|Tell HN:|Launch HN:)\s*', '', str(text), flags=re.IGNORECASE)
    return text.strip()

df['clean_title'] = df['Title'].apply(clean_title_text)

# 📌 5. ลบข้อมูลซ้ำ (Drop Duplicates)
# เก็บแถวแรกไว้ (keep='first')
df_clean = df.drop_duplicates(subset=['clean_title', 'language'], keep='first')
dropped_rows = len(df) - len(df_clean)

# ==========================================
# 📊 6. DATA VALIDATION REPORT
# ==========================================

print("\n" + "="*45)
print(" 📊 DATA VALIDATION REPORT (ผลการตรวจสอบ)")
print("="*45)

# มิติที่ 1: Completeness (ความครบถ้วน)
print("\n1️⃣ ความครบถ้วน (Completeness):")
print(f"   🔹 ข้อมูลเริ่มต้น: {initial_rows} แถว")
print(f"   🔹 ลบข้อมูลที่ซ้ำซ้อนออก: {dropped_rows} แถว")
print(f"   🔹 ข้อมูลสุทธิพร้อมใช้งาน: {len(df_clean)} แถว")
missing_titles = df_clean['clean_title'].isnull().sum()
print(f"   🔹 ค่าว่าง (Null) ในชื่อกระทู้: {missing_titles} แถว (ผ่านเงื่อนไข = {missing_titles == 0})")

# มิติที่ 2: Uniqueness (ความไม่ซ้ำซ้อน)
print("\n2️⃣ ความไม่ซ้ำซ้อน (Uniqueness):")
duplicates_left = df_clean.duplicated(subset=['clean_title', 'language']).sum()
print(f"   🔹 ข้อมูลซ้ำที่หลงเหลือในระบบ: {duplicates_left} แถว (ผ่านเงื่อนไข = {duplicates_left == 0})")

# มิติที่ 3: Consistency (ความสม่ำเสมอของหมวดหมู่)
print("\n3️⃣ ความสม่ำเสมอของข้อมูล (Consistency):")
invalid_langs = ['Golang', 'Dlang', 'PL/SQL', 'Classic Visual Basic']
found_invalid = df_clean[df_clean['language'].isin(invalid_langs)]
print(f"   🔹 ชื่อภาษาขยะที่ตกหล่น: {len(found_invalid)} แถว (ผ่านเงื่อนไข = {len(found_invalid) == 0})")
print("-" * 45)

# 📌 7. บันทึกเป็นไฟล์ JSON (จัดฟอร์แมตให้อ่านง่ายและรองรับอักขระพิเศษ)
output_dir = '../raw_data'
output_file = 'Final_HackerNews_Cleaned.json'
output_path = os.path.join(output_dir, output_file)

os.makedirs(output_dir, exist_ok=True) 
df_clean = df[['clean_title','language','Scrap_Year','Actual_Post_Year']]
# เซฟไฟล์พร้อมจัดฟอร์แมตให้สวยงามเหมือนต้นฉบับ
df_clean.to_json(output_path, orient='records', force_ascii=False, indent=4)

print(f"\n✅ บันทึกไฟล์เสร็จสมบูรณ์ที่: {output_path}")


 📊 DATA VALIDATION REPORT (ผลการตรวจสอบ)

1️⃣ ความครบถ้วน (Completeness):
   🔹 ข้อมูลเริ่มต้น: 37224 แถว
   🔹 ลบข้อมูลที่ซ้ำซ้อนออก: 5724 แถว
   🔹 ข้อมูลสุทธิพร้อมใช้งาน: 32528 แถว
   🔹 ค่าว่าง (Null) ในชื่อกระทู้: 0 แถว (ผ่านเงื่อนไข = True)

2️⃣ ความไม่ซ้ำซ้อน (Uniqueness):
   🔹 ข้อมูลซ้ำที่หลงเหลือในระบบ: 0 แถว (ผ่านเงื่อนไข = True)

3️⃣ ความสม่ำเสมอของข้อมูล (Consistency):
   🔹 ชื่อภาษาขยะที่ตกหล่น: 0 แถว (ผ่านเงื่อนไข = True)
---------------------------------------------

✅ บันทึกไฟล์เสร็จสมบูรณ์ที่: ../raw_data\Final_HackerNews_Cleaned.json


In [7]:
# Data Transfrom
import pandas as pd
import os
import json

file_adzu = 'Adzuna_Cleaned.json'
file_dice = 'cleaned_jobs_data_dice.json'
file_thaijob = 'cleaned_jobs.json'
file_hacknews = 'Final_Final_HackerNews_Cleaned.json'
file_survey = 'So_survey_total_prolang_desire.json'
output_dir = '../raw_data'
os.makedirs(output_dir, exist_ok=True) 
output_path = os.path.join(output_dir, file_adzu)
with open(output_path, 'r', encoding='utf-8') as f:
        df_adzuna = pd.DataFrame(json.load(f))

output_path = os.path.join(output_dir, file_dice)
with open(output_path, 'r', encoding='utf-8') as f:
        df_dice = pd.DataFrame(json.load(f))
output_path = os.path.join(output_dir, file_thaijob)
with open(output_path, 'r', encoding='utf-8') as f:
        df_thaijob = pd.DataFrame(json.load(f))
output_path = os.path.join(output_dir, file_hacknews)
with open(output_path, 'r', encoding='utf-8') as f:
        df_hacknews = pd.DataFrame(json.load(f))
output_path = os.path.join(output_dir, file_survey)
with open(output_path, 'r', encoding='utf-8') as f:
        df_survey = pd.DataFrame(json.load(f))



In [8]:
print(f'=== Adzuna ===\nFind Data: {len(df_adzuna)} record\n')
print(df_adzuna.info())
print(f'=== Dice ===\nFind Data: {len(df_dice)} record\n')
print(df_dice.info())
print(f'=== ThaiJob ===\nFind Data: {len(df_thaijob)} record\n')
print(df_thaijob.info())
print(f'=== HackerNews ===\nFind Data: {len(df_hacknews)} record\n')
print(df_hacknews.info())
print(f'=== Survey ===\nFind Data: {len(df_survey)} record\n')
print(df_survey.info())

=== Adzuna ===
Find Data: 4508 record

<class 'pandas.DataFrame'>
RangeIndex: 4508 entries, 0 to 4507
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         4508 non-null   int64 
 1   company    4485 non-null   str   
 2   title      4508 non-null   str   
 3   year       4508 non-null   int64 
 4   languages  4508 non-null   object
dtypes: int64(2), object(1), str(2)
memory usage: 176.2+ KB
None
=== Dice ===
Find Data: 12519 record

<class 'pandas.DataFrame'>
RangeIndex: 12519 entries, 0 to 12518
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   post_id             12519 non-null  str  
 1   company_name        12519 non-null  str  
 2   job_name            12519 non-null  str  
 3   skills              12519 non-null  str  
 4   date_scraped        12519 non-null  str  
 5   date_posted         12519 non-null  str  
 6   actual_date

In [9]:
df_adzuna = df_adzuna.rename(columns={'languages': 'Language'})
df_adzuna['Language'] = df_adzuna['Language'].astype('str')
df_dice = df_dice.rename(columns={'base_language': 'Language','company_name': 'company'})


In [10]:
display(df_adzuna.info())
display(df_dice.info())

<class 'pandas.DataFrame'>
RangeIndex: 4508 entries, 0 to 4507
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        4508 non-null   int64
 1   company   4485 non-null   str  
 2   title     4508 non-null   str  
 3   year      4508 non-null   int64
 4   Language  4508 non-null   str  
dtypes: int64(2), str(3)
memory usage: 176.2 KB


None

<class 'pandas.DataFrame'>
RangeIndex: 12519 entries, 0 to 12518
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   post_id             12519 non-null  str  
 1   company             12519 non-null  str  
 2   job_name            12519 non-null  str  
 3   skills              12519 non-null  str  
 4   date_scraped        12519 non-null  str  
 5   date_posted         12519 non-null  str  
 6   actual_date_posted  12519 non-null  str  
 7   tech_name           12519 non-null  str  
 8   tech_category       12519 non-null  str  
 9   Language            12519 non-null  str  
dtypes: str(10)
memory usage: 978.2 KB


None

In [13]:
df_merge_job = pd.merge(
    left= df_adzuna,
    right= df_dice,
    on='Language',
    how= 'outer'
)

display(df_merge_job.info())

<class 'pandas.DataFrame'>
RangeIndex: 17027 entries, 0 to 17026
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  4508 non-null   float64
 1   company_x           4485 non-null   str    
 2   title               4508 non-null   str    
 3   year                4508 non-null   float64
 4   Language            17027 non-null  str    
 5   post_id             12519 non-null  str    
 6   company_y           12519 non-null  str    
 7   job_name            12519 non-null  str    
 8   skills              12519 non-null  str    
 9   date_scraped        12519 non-null  str    
 10  date_posted         12519 non-null  str    
 11  actual_date_posted  12519 non-null  str    
 12  tech_name           12519 non-null  str    
 13  tech_category       12519 non-null  str    
dtypes: float64(2), str(12)
memory usage: 1.8 MB


None

In [15]:
df_merge_job['company'] = df_merge_job['company_x'].fillna(df_merge_job['company_y'])
display(df_merge_job.head(3))

,id,company_x,title,year,Language,post_id,company_y,job_name,skills,date_scraped,date_posted,actual_date_posted,tech_name,tech_category,company
0,NaN,NaN,NaN,NaN,Bash & Shell,GLOBINVA,"Global Infotek, Inc.",CNO Developer/Engineer - JBMD 1635,Bash,2026-02-12T00:00:00.000,1 day ago,2026-02-11T00:00:00.000,BASH,Programming Language,"Global Infotek, Inc."
1,NaN,NaN,NaN,NaN,Bash & Shell,91139932,Clover Solutions LLC,Sr. Camera Validation Engineer,Bash,2026-02-12T00:00:00.000,3 days ago,2026-02-09T00:00:00.000,BASH,Programming Language,Clover Solutions LLC
2,NaN,NaN,NaN,NaN,Bash & Shell,10112570,ASSYST,Oracle Technical Specialist,Shell,2026-02-12T00:00:00.000,22 days ago,2026-01-21T00:00:00.000,SHELL,Programming Language,ASSYST
